# MovieLens 100K initial experiments

This notebook runs ItemKNN and BPR and produces top-10 recommendations for a representative user. LightGCN is skipped for now because PyTorch warned about possible memory errors while constructing its sparse graph tensor. The run summary, CSV outputs, and README describe the current run.

An optional API-backed LLM reranker reorders the top 100 BPR candidates. It needs `LLM_API_KEY` (or `OPENAI_API_KEY`); without a configured key, it reports a skipped status and does not claim an LLM recommendation list.

## 1. Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

HERE = Path.cwd()
if not (HERE / "run_initial_experiments.py").exists():
    HERE = HERE / "Initial Experiments"
sys.path.insert(0, str(HERE))
from run_initial_experiments import run_experiment, OUT_DIR

print("Runner:", HERE / "run_initial_experiments.py")
print("LightGCN disabled for this run.")


Runner: D:\MSc TUD\MSc Thesis\Masters_Thesis\Initial Experiments\run_initial_experiments.py
LightGCN disabled for this run.


## 2. Run ItemKNN and BPR

In [2]:
# Runs ItemKNN and BPR. LightGCN is disabled while investigating its sparse-tensor warning.
# The LLM reranker runs only when LLM_API_KEY or OPENAI_API_KEY is configured.
summary, top_lists = run_experiment()


{
  "dataset": "MovieLens 100K prepared RecBole TSV",
  "interactions": 100000,
  "users": 943,
  "items": 1682,
  "positive_threshold": 4.0,
  "training_positive_interactions": 53603,
  "heldout_users": 942,
  "split": "per-user latest positive held out; each evaluated user's events at or after that timestamp are censored from training",
  "example_user_id": 450,
  "example_user_training_positive_count": 377,
  "example_user_top_genres": [
    [
      "Drama",
      179
    ],
    [
      "Comedy",
      111
    ],
    [
      "Romance",
      93
    ],
    [
      "Thriller",
      67
    ],
    [
      "Action",
      62
    ],
    [
      "Adventure",
      40
    ],
    [
      "Children's",
      32
    ],
    [
      "War",
      30
    ]
  ],
  "metrics": {
    "ItemKNN": {
      "HR@10": 0.10084925690021232,
      "NDCG@10": 0.05575681324473098,
      "evaluated_users": 942,
      "median_target_rank": 144.5
    },
    "BPR": {
      "HR@10": 0.12314225053078556,
      "NDCG@1

## 3. Evaluation

In [3]:
metrics = pd.read_csv(OUT_DIR / "metrics.csv")
display(metrics)


algorithm,HR@10,NDCG@10,evaluated_users,median_target_rank
ItemKNN,0.100849,0.055757,942,144.5
BPR,0.123142,0.061495,942,92.5


The example top-10 lists below are for MovieLens user **450**. Model scores have different scales, so compare ranks and item lists rather than raw scores.

In [4]:
recommendations = pd.read_csv(OUT_DIR / "top10_recommendations.csv")
display(recommendations[["algorithm", "rank", "movie_title", "genres", "score", "user_id"]])


algorithm,rank,movie_title,genres,score,user_id
ItemKNN,1,Babe,Children's Comedy Drama,87.176979,450
ItemKNN,2,This Is Spinal Tap,Comedy Drama Musical,77.898964,450
ItemKNN,3,Crimson Tide,Drama Thriller War,77.050713,450
ItemKNN,4,Brazil,Sci-Fi,76.076714,450
ItemKNN,5,Dead Man Walking,Drama,76.053337,450
ItemKNN,6,In the Line of Fire,Action Thriller,76.010223,450
ItemKNN,7,Pinocchio,Animation Children's,69.889084,450
ItemKNN,8,Hoop Dreams,Documentary,67.602333,450
ItemKNN,9,"Madness of King George, The",Drama,66.140846,450
ItemKNN,10,Clerks,Comedy,66.064682,450


## 4. How the algorithms differ

- **ItemKNN** scores unseen movies by their similarity to movies the user liked. Its recommendations are based on local item-to-item overlap; it is simple and can be explained through similar items, but sparse histories and popularity can limit it.
- **BPR** learns user and item factors by ranking observed positives above sampled negatives. It can generalize beyond direct item neighbors; results depend on negative sampling and regularization.
- **LightGCN** propagates user and item embeddings across the interaction graph and combines embeddings from multiple graph layers. It is skipped in this run due to the PyTorch sparse-tensor warning.
- **LLM reranking** uses BPR to retrieve 100 candidates, then asks a configured LLM to reorder those candidates using titles, genres, and the user's genre history. It cannot recommend a relevant movie that BPR omitted from the candidate pool. The LLM call was skipped here because no API endpoint/key was configured.

On the current one-target-per-user split, BPR leads ItemKNN on HR@10 and NDCG@10. These are starter settings, so treat the gap as a direction for follow-up work rather than a final conclusion.

## 5. Top-10 list overlap

In [5]:
overlap = pd.read_csv(OUT_DIR / "top10_overlap.csv")
display(overlap)


algorithm_a,algorithm_b,shared_items,jaccard_at_10
ItemKNN,BPR,3,0.176471


See `README.md` for split details and rerun instructions. Outputs are saved under `outputs/`.